# UAVDT Notebook 03 — Project Vehicles into BEV and Build Simple 3D Cuboids

Continues from Notebook 02. It uses the saved image-to-BEV homography to detect or load vehicles, project them to BEV, create lightweight tracks, estimate heading/speed in BEV pixel units, and render simple 3D cuboids.

This is a fast BEV-style reconstruction prototype, not photogrammetric 3D reconstruction.


In [ ]:
#@title 1. Set local project paths

from pathlib import Path
import json
import os
import shutil
import math
import csv
import warnings
from notebook_local import resolve_project_dir, print_local_setup

warnings.filterwarnings('ignore')

PROJECT_DIR = resolve_project_dir()
SAMPLE_DIR = PROJECT_DIR / 'work' / 'M1401_sample' / 'images'
HOMOGRAPHY_DIR = PROJECT_DIR / 'work' / 'notebook_02_bev_homography'
OUTPUT_DIR = PROJECT_DIR / 'work' / 'notebook_03_bev_vehicles'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print_local_setup(PROJECT_DIR)
print('SAMPLE_DIR:', SAMPLE_DIR)
print('HOMOGRAPHY_DIR:', HOMOGRAPHY_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Sample image folder exists:', SAMPLE_DIR.exists())
print('Homography folder exists:', HOMOGRAPHY_DIR.exists())


In [ ]:
#@title 2. Import dependencies

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print('cv2:', cv2.__version__)
print('numpy:', np.__version__)
print('pandas:', pd.__version__)


In [ ]:
#@title 3. Load frames and homography from Notebook 02

from pathlib import Path
import json
import numpy as np

image_exts = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
frames = sorted([p for p in SAMPLE_DIR.iterdir() if p.suffix in image_exts])
print('Frames found:', len(frames))
if len(frames) == 0:
    raise RuntimeError('No frames found. Run Notebook 01 first and create M1401_sample/images.')

candidate_h_files = [
    HOMOGRAPHY_DIR / 'homography.json',
    HOMOGRAPHY_DIR / 'bev_homography.json',
    HOMOGRAPHY_DIR / 'image_to_bev_homography.json',
]

homography_file = None
for p in candidate_h_files:
    if p.exists():
        homography_file = p
        break

if homography_file is None:
    json_files = sorted(HOMOGRAPHY_DIR.glob('*.json'))
    print('JSON files found in homography dir:', [p.name for p in json_files])
    if len(json_files) == 0:
        raise RuntimeError('No homography JSON found. Run Notebook 02 first.')
    homography_file = json_files[0]

print('Using homography file:', homography_file)
with open(homography_file, 'r') as f:
    hdata = json.load(f)

H = None
for key in ['H_image_to_bev', 'homography', 'H', 'image_to_bev']:
    if isinstance(hdata, dict) and key in hdata:
        H = np.array(hdata[key], dtype=np.float32)
        break
if H is None and isinstance(hdata, list):
    H = np.array(hdata, dtype=np.float32)
if H is None or H.shape != (3, 3):
    raise RuntimeError('Could not read a 3x3 homography matrix from JSON.')

bev_width = int(hdata.get('bev_width', 500)) if isinstance(hdata, dict) else 500
bev_height = int(hdata.get('bev_height', 900)) if isinstance(hdata, dict) else 900

print('H matrix:')
print(H)
print('BEV size:', bev_width, bev_height)
print('First frame:', frames[0].name)
print('Last frame:', frames[-1].name)


In [ ]:
#@title 4. Preview BEV warp

img = cv2.imread(str(frames[0]))
if img is None:
    raise RuntimeError('Could not read first frame.')

bev = cv2.warpPerspective(img, H, (bev_width, bev_height))
preview_path = OUTPUT_DIR / 'bev_reference_preview.jpg'
cv2.imwrite(str(preview_path), bev)

plt.figure(figsize=(6, 9))
plt.imshow(cv2.cvtColor(bev, cv2.COLOR_BGR2RGB))
plt.title('BEV warp from Notebook 02 homography')
plt.axis('off')
plt.show()
print('Saved:', preview_path)


## Detection source

Use `yolo` for a fully unmanned first run. Use `csv` if you already have UAVDT labels/detections in a CSV with columns: `frame`, `x1`, `y1`, `x2`, `y2`, and optionally `class_name`, `track_id`, `confidence`.


In [ ]:
#@title 5. Detect vehicles or load detections

DETECTION_SOURCE = 'yolo' #@param ['yolo', 'csv']
CSV_DETECTIONS_PATH = '' #@param {type:'string'}
MAX_FRAMES = 30 #@param {type:'integer'}
YOLO_MODEL = 'yolov8n.pt' #@param ['yolov8n.pt', 'yolov8s.pt']
CONF_THRES = 0.25 #@param {type:'number'}

selected_frames = frames[:max(1, min(MAX_FRAMES, len(frames)))]
print('Selected frames for Notebook 03:', len(selected_frames))

vehicle_class_names = {'car', 'truck', 'bus', 'motorcycle'}
detections_csv = OUTPUT_DIR / 'detections_image_space.csv'

if DETECTION_SOURCE == 'csv':
    if not CSV_DETECTIONS_PATH:
        raise RuntimeError('CSV_DETECTIONS_PATH is empty.')
    src_csv = Path(CSV_DETECTIONS_PATH)
    if not src_csv.exists():
        raise RuntimeError('CSV_DETECTIONS_PATH does not exist: ' + str(src_csv))
    df_det = pd.read_csv(src_csv)
    required = {'frame', 'x1', 'y1', 'x2', 'y2'}
    missing = required - set(df_det.columns)
    if missing:
        raise RuntimeError('Detection CSV missing required columns: ' + str(sorted(missing)))
    if 'frame_idx' not in df_det.columns:
        frame_to_idx = {p.name: i for i, p in enumerate(selected_frames)}
        df_det['frame_idx'] = df_det['frame'].map(frame_to_idx)
        df_det = df_det[df_det['frame_idx'].notna()].copy()
        df_det['frame_idx'] = df_det['frame_idx'].astype(int)
    if 'class_name' not in df_det.columns:
        df_det['class_name'] = 'vehicle'
    if 'confidence' not in df_det.columns:
        df_det['confidence'] = 1.0
    df_det.to_csv(detections_csv, index=False)
    print('Loaded detections:', len(df_det))
else:
    from ultralytics import YOLO
    model = YOLO(YOLO_MODEL)
    rows = []
    for frame_idx, frame_path in enumerate(selected_frames):
        result = model.predict(str(frame_path), conf=CONF_THRES, verbose=False)[0]
        names = result.names
        if result.boxes is None:
            continue
        for box in result.boxes:
            cls_id = int(box.cls.item())
            class_name = names.get(cls_id, str(cls_id))
            if class_name not in vehicle_class_names:
                continue
            x1, y1, x2, y2 = box.xyxy[0].detach().cpu().numpy().tolist()
            conf = float(box.conf.item())
            rows.append({
                'frame_idx': frame_idx,
                'frame': frame_path.name,
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
                'class_name': class_name,
                'confidence': conf,
            })
    df_det = pd.DataFrame(rows)
    df_det.to_csv(detections_csv, index=False)
    print('YOLO detections:', len(df_det))

print('Saved detections to:', detections_csv)
display(df_det.head(20))


In [ ]:
#@title 6. Project vehicle detections into BEV

bev_positions_csv = OUTPUT_DIR / 'vehicle_bev_positions.csv'

def project_points(points_xy, H):
    pts = np.array(points_xy, dtype=np.float32).reshape(-1, 1, 2)
    out = cv2.perspectiveTransform(pts, H).reshape(-1, 2)
    return out

if len(df_det) == 0:
    raise RuntimeError('No vehicle detections available. Try lowering CONF_THRES or using a label CSV.')

df_bev = df_det.copy()
df_bev['ground_u'] = 0.5 * (df_bev['x1'] + df_bev['x2'])
df_bev['ground_v'] = df_bev['y2']
bev_xy = project_points(df_bev[['ground_u', 'ground_v']].values, H)
df_bev['bev_x'] = bev_xy[:, 0]
df_bev['bev_y'] = bev_xy[:, 1]

margin = 20
inside = (
    (df_bev['bev_x'] >= -margin) &
    (df_bev['bev_x'] <= bev_width + margin) &
    (df_bev['bev_y'] >= -margin) &
    (df_bev['bev_y'] <= bev_height + margin)
)
df_bev = df_bev[inside].copy().reset_index(drop=True)

df_bev.to_csv(bev_positions_csv, index=False)
print('Projected BEV detections:', len(df_bev))
print('Saved:', bev_positions_csv)
display(df_bev.head(20))


In [ ]:
#@title 7. Simple nearest-neighbor tracking in BEV

MAX_LINK_DISTANCE = 80.0 #@param {type:'number'}

if 'track_id' in df_bev.columns and df_bev['track_id'].notna().any():
    print('Using existing track_id column from detections.')
    df_tracks = df_bev.copy()
else:
    next_track_id = 1
    active_tracks = {}
    assigned_rows = []
    for frame_idx in sorted(df_bev['frame_idx'].unique()):
        frame_rows = df_bev[df_bev['frame_idx'] == frame_idx].copy()
        used_tracks = set()
        for row_idx, row in frame_rows.iterrows():
            p = np.array([row['bev_x'], row['bev_y']], dtype=float)
            best_id = None
            best_dist = float('inf')
            for tid, last in active_tracks.items():
                if tid in used_tracks:
                    continue
                if frame_idx - last['frame_idx'] > 2:
                    continue
                q = np.array([last['bev_x'], last['bev_y']], dtype=float)
                dist = float(np.linalg.norm(p - q))
                if dist < best_dist:
                    best_dist = dist
                    best_id = tid
            if best_id is not None and best_dist <= MAX_LINK_DISTANCE:
                tid = best_id
            else:
                tid = next_track_id
                next_track_id += 1
            used_tracks.add(tid)
            active_tracks[tid] = {'frame_idx': frame_idx, 'bev_x': row['bev_x'], 'bev_y': row['bev_y']}
            r = row.to_dict()
            r['track_id'] = tid
            r['link_distance'] = best_dist if best_id is not None else None
            assigned_rows.append(r)
    df_tracks = pd.DataFrame(assigned_rows)

tracks_csv = OUTPUT_DIR / 'vehicle_bev_tracks.csv'
df_tracks.to_csv(tracks_csv, index=False)
print('Tracks:', df_tracks['track_id'].nunique())
print('Tracked detections:', len(df_tracks))
print('Saved:', tracks_csv)
display(df_tracks.head(20))


In [ ]:
#@title 8. Estimate heading and speed in BEV pixels

FPS = 30.0 #@param {type:'number'}
FRAME_STRIDE = 1 #@param {type:'integer'}

rows = []
for tid, g in df_tracks.groupby('track_id'):
    g = g.sort_values('frame_idx').copy()
    xs = g['bev_x'].values
    ys = g['bev_y'].values
    fis = g['frame_idx'].values
    yaw = np.zeros(len(g), dtype=float)
    speed = np.zeros(len(g), dtype=float)
    for i in range(len(g)):
        if len(g) == 1:
            dx, dy = 1.0, 0.0
        elif i == 0:
            dx = xs[i+1] - xs[i]
            dy = ys[i+1] - ys[i]
        else:
            dx = xs[i] - xs[i-1]
            dy = ys[i] - ys[i-1]
        yaw[i] = math.atan2(dy, dx)
        if i > 0:
            dt = max(1e-6, (fis[i] - fis[i-1]) * FRAME_STRIDE / FPS)
            speed[i] = math.hypot(dx, dy) / dt
    g['yaw_rad'] = yaw
    g['speed_bev_px_per_s'] = speed
    rows.append(g)

df_state = pd.concat(rows, ignore_index=True) if rows else df_tracks.copy()
state_csv = OUTPUT_DIR / 'vehicle_bev_states.csv'
df_state.to_csv(state_csv, index=False)
print('Saved:', state_csv)
display(df_state.head(20))


In [ ]:
#@title 9. Render BEV vehicle positions and trajectories

bev_overlay_dir = OUTPUT_DIR / 'bev_overlays'
if bev_overlay_dir.exists():
    shutil.rmtree(bev_overlay_dir)
bev_overlay_dir.mkdir(parents=True, exist_ok=True)

ref_img = cv2.imread(str(selected_frames[0]))
bev_bg = cv2.warpPerspective(ref_img, H, (bev_width, bev_height))

def color_for_id(tid):
    rng = np.random.default_rng(int(tid) * 1009)
    c = rng.integers(40, 255, size=3)
    return int(c[0]), int(c[1]), int(c[2])

for frame_idx in sorted(df_state['frame_idx'].unique()):
    canvas = bev_bg.copy()
    past = df_state[df_state['frame_idx'] <= frame_idx]
    current = df_state[df_state['frame_idx'] == frame_idx]
    for tid, g in past.groupby('track_id'):
        pts = g.sort_values('frame_idx')[['bev_x', 'bev_y']].values.astype(int)
        color = color_for_id(tid)
        for i in range(1, len(pts)):
            cv2.line(canvas, tuple(pts[i-1]), tuple(pts[i]), color, 2)
    for _, row in current.iterrows():
        x = int(row['bev_x'])
        y = int(row['bev_y'])
        tid = int(row['track_id'])
        color = color_for_id(tid)
        cv2.circle(canvas, (x, y), 6, color, -1)
        cv2.putText(canvas, str(tid), (x + 7, y - 7), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    out_path = bev_overlay_dir / f'bev_overlay_{int(frame_idx):05d}.jpg'
    cv2.imwrite(str(out_path), canvas)

print('Saved BEV overlays to:', bev_overlay_dir)
sample_overlay = sorted(bev_overlay_dir.glob('*.jpg'))[-1]
img_show = cv2.imread(str(sample_overlay))
plt.figure(figsize=(6, 9))
plt.imshow(cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB))
plt.title('BEV tracks overlay')
plt.axis('off')
plt.show()


In [ ]:
#@title 10. Build simple 3D cuboids for vehicles

CLASS_DIMS = {
    'car':        {'length': 45.0, 'width': 18.0, 'height': 15.0},
    'truck':      {'length': 80.0, 'width': 25.0, 'height': 30.0},
    'bus':        {'length': 110.0, 'width': 26.0, 'height': 32.0},
    'motorcycle': {'length': 25.0, 'width': 8.0,  'height': 12.0},
    'vehicle':    {'length': 45.0, 'width': 18.0, 'height': 15.0},
}

def dims_for_class(name):
    return CLASS_DIMS.get(str(name), CLASS_DIMS['vehicle'])

cuboids = []
for _, row in df_state.iterrows():
    dims = dims_for_class(row.get('class_name', 'vehicle'))
    cuboids.append({
        'frame_idx': int(row['frame_idx']),
        'frame': str(row['frame']),
        'track_id': int(row['track_id']),
        'class_name': str(row.get('class_name', 'vehicle')),
        'center_x': float(row['bev_x']),
        'center_y': float(row['bev_y']),
        'center_z': float(dims['height'] / 2.0),
        'length': float(dims['length']),
        'width': float(dims['width']),
        'height': float(dims['height']),
        'yaw_rad': float(row['yaw_rad']),
        'speed_bev_px_per_s': float(row['speed_bev_px_per_s']),
    })

cuboids_json = OUTPUT_DIR / 'vehicle_3d_cuboids_bev_pixels.json'
with open(cuboids_json, 'w') as f:
    json.dump({'coordinate_system': 'BEV pixels', 'cuboids': cuboids}, f, indent=2)

print('Cuboids:', len(cuboids))
print('Saved:', cuboids_json)
print('Example cuboid:')
print(json.dumps(cuboids[0], indent=2) if cuboids else 'No cuboids')


In [ ]:
#@title 11. Render simple 3D scene preview with cuboids

from mpl_toolkits.mplot3d.art3d import Poly3DCollection

PREVIEW_FRAME_IDX = int(df_state['frame_idx'].max()) if len(df_state) else 0
frame_cuboids = [c for c in cuboids if c['frame_idx'] == PREVIEW_FRAME_IDX]
print('Preview frame idx:', PREVIEW_FRAME_IDX)
print('Cuboids in preview:', len(frame_cuboids))

def cuboid_vertices(cx, cy, cz, length, width, height, yaw):
    l = length / 2.0
    w = width / 2.0
    h = height / 2.0
    base = np.array([
        [-l, -w, -h], [ l, -w, -h], [ l,  w, -h], [-l,  w, -h],
        [-l, -w,  h], [ l, -w,  h], [ l,  w,  h], [-l,  w,  h],
    ], dtype=float)
    R = np.array([
        [math.cos(yaw), -math.sin(yaw), 0],
        [math.sin(yaw),  math.cos(yaw), 0],
        [0, 0, 1],
    ], dtype=float)
    return base @ R.T + np.array([cx, cy, cz])

def cuboid_faces(verts):
    return [
        [verts[i] for i in [0, 1, 2, 3]],
        [verts[i] for i in [4, 5, 6, 7]],
        [verts[i] for i in [0, 1, 5, 4]],
        [verts[i] for i in [1, 2, 6, 5]],
        [verts[i] for i in [2, 3, 7, 6]],
        [verts[i] for i in [3, 0, 4, 7]],
    ]

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, projection='3d')
ax.plot([0, bev_width, bev_width, 0, 0], [0, 0, bev_height, bev_height, 0], [0, 0, 0, 0, 0])

for c in frame_cuboids:
    verts = cuboid_vertices(c['center_x'], c['center_y'], c['center_z'], c['length'], c['width'], c['height'], c['yaw_rad'])
    faces = cuboid_faces(verts)
    poly = Poly3DCollection(faces, alpha=0.35, linewidths=0.5, edgecolors='k')
    ax.add_collection3d(poly)
    ax.text(c['center_x'], c['center_y'], c['height'] + 5, str(c['track_id']), fontsize=8)

ax.set_xlim(0, bev_width)
ax.set_ylim(bev_height, 0)
ax.set_zlim(0, 80)
ax.set_xlabel('BEV x px')
ax.set_ylabel('BEV y px')
ax.set_zlabel('height px')
ax.set_title('Simple dynamic 3D scene: road plane + vehicle cuboids')

preview_3d_path = OUTPUT_DIR / 'simple_3d_cuboids_preview.png'
plt.savefig(preview_3d_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', preview_3d_path)


In [ ]:
#@title 12. Summary of outputs for Notebook 04

print('Notebook 03 outputs:')
for p in [
    detections_csv,
    bev_positions_csv,
    tracks_csv,
    state_csv,
    cuboids_json,
    preview_path,
    OUTPUT_DIR / 'simple_3d_cuboids_preview.png',
]:
    print('-', p, 'exists:', Path(p).exists())

print('BEV overlays folder:', bev_overlay_dir)
print('Number of overlay frames:', len(list(bev_overlay_dir.glob('*.jpg'))))
print('Next notebook idea:')
print('Notebook 04 can calibrate BEV pixels to meters using lane width or car-size priors, then export the dynamic scene to glTF/OBJ or render a video.')
